In [1]:
import os
import pathlib
import sys


DNSCBOR_EVAL_DIR = pathlib.Path.cwd()
os.environ["DNSCBOR_EVAL_DIR"] = str(DNSCBOR_EVAL_DIR)

if str((DNSCBOR_EVAL_DIR / "..").absolute()) not in sys.path:
    sys.path.append(str((DNSCBOR_EVAL_DIR / "..").absolute()))

from utils import list_code

# application/dns+cbor: Dataset Collection

## MassDNS Tranco List at Public Resolvers

We use the [MassDNS] tool by Birk Blechschmidt and Quirin Scheitle to query AAAA, A, HTTPS, NS, PTR, DS, RRSIG, DNSKEY, NSEC, and NSEC3 records for the names in the Tranco list from the following public resolvers:

- `1.1.1.1` (CloudFlare)
- `8.8.8.8` (Google)
- `9.9.9.9` (Quad9)

To build MassDNS, use the following command.

[MassDNS]: https://github.com/blechschmidt/massdns

In [2]:
%%bash
git -C "${DNSCBOR_EVAL_DIR}"/.. submodule update --init
make -C "${DNSCBOR_EVAL_DIR}/massdns"

Submodule path '04_cbor4dns_eval/massdns': checked out 'bad45b873057637ae69b9d9f4c1c179126f97a48'
make: Entering directory '/app/04_cbor4dns_eval/massdns'
mkdir -p bin
cc    -DMASSDNS_REVISION=\"v1.1.0-7-gbad45b8\" -O3 -std=c11 -DHAVE_EPOLL -DHAVE_SYSINFO -Wall -fstack-protector-strong src/main.c -o bin/massdns
make: Leaving directory '/app/04_cbor4dns_eval/massdns'


We use the following to generate the DNS traffic. **This may run for a while and may use up your bandwidth significantly; we recommend running it off-site or while you are asleep.** We provide a complete PCAP on [OPARA](https://doi.org/10.25532/OPARA-1530) if you do not have the resources to generate it.

In [3]:
list_code(DNSCBOR_EVAL_DIR / "massdns_resolve.sh")

#!/bin/bash

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

EXP="${EXP:-}"
MASSDNS_DIR="${SCRIPT_DIR}/massdns"
RESULTS_DIR="${RESULTS_DIR:-${SCRIPT_DIR}/input_datasets/tranco}"
PROCS="$(grep -c '^processor' /proc/cpuinfo)"
TRANCO_SET=KJ49W
RESOLVERS=( "8.8.8.8" "1.1.1.1" "9.9.9.9" )
CNAME_ITERATION=1
RRTYPES="-t AAAA -t A -t HTTPS -t NS -t PTR -t DS -t RRSIG -t DNSKEY -t NSEC -t NSEC3"
RRTYPES_NAMES=$(echo "${RRTYPES}" | sed -E 's/ ?-t /_/g')

if [ $# -gt 0 ]; then
    SNIFF_IFACE=$1
fi

mkdir -p "${RESULTS_DIR}"

if [ ! -f "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" ]; then
    wget -O "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" "https://tranco-list.eu/download/${TRANCO_SET}/full" \
        2>&1 || exit 1
fi

echo -n "" > "${SCRIPT_DIR}/resolvers.txt"

for RESOLVER in ${RESOLVERS[@]}; do
    echo "${RESOLVER}" >> "${SCRIPT_DIR}/resolvers.txt"
done

if [ -n "${SNIFF_IFACE}" ]; then
    tshark -i "${SNIFF_IFACE}" \
        -w "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}.pcapng" "port 53" &
    TSHARK_PID=$!
    sleep 5  # make sure tshark listens
fi

awk -F, '{print $2}' "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" | sed 's/\r$//' |
    "${MASSDNS_DIR}/bin/massdns" --status-format json -r "${SCRIPT_DIR}/resolvers.txt" ${RRTYPES} -o J | \
    sed "s/}\$/,\"cname_iteration\":${CNAME_ITERATION}}/" \
    > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson"

while grep -q -e "CNAME" \
        "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson"; do
     CNAME_ITERATION=$((CNAME_ITERATION + 1))
    cat "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_$(( CNAME_ITERATION - 1 )).ndjson" | \
    jq -r '.data[][] | select (.type == "CNAME") | .data' | sort -u | \
        "${MASSDNS_DIR}/bin/massdns" --status-format json -r "${SCRIPT_DIR}/resolvers.txt" ${RRTYPES} -o J | \
    sed "s/}\$/,\"cname_iteration\":${CNAME_ITERATION}}/" \
        > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson" \

    if [ ${CNAME_ITERATION} -eq 10 ]; then
         break
    fi
done

if [ -n "${TSHARK_PID}" ]; then
    kill "${TSHARK_PID}"
fi

cat "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_"*.ndjson | xz \
    > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}.ndjson.xz"

You need the network interface you want to query over. This is normally something like `eno1` or `eth0`. Use the following command to find it. The one that is **not** `lo`, is the one you want.

```console
$ ip link
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: eno0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether C0:15:51:D9:0D:94 brd ff:ff:ff:ff:ff:ff
    altname enxc01551d9ad94
```

You also need the a full tranco list. The one one we used for our setup, the one from January 11, 2024, you can get [on the Tranco website](https://tranco-list.eu/list/KJ49W/full). Put the downloaded CSV file into into `04_cbor4dns_eval/input_datasets/tranco/tranco_KJ49W_full.csv`. If you use a different one, you have to change the `TRANCO_SET` variable in [the shell script shown above](./massdns_resolve.sh) to the ID of that list first, before running the next command.

In [4]:
%%bash
tmux new-session -s "massdns" -d "'${DNSCBOR_EVAL_DIR}'/massdns_resolve.sh eth0 2> '${DNSCBOR_EVAL_DIR}'/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.status.txt"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "massdns"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "massdns" C-c
```

Errors are logged within `04_cbor4dns_eval/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.status.txt` (the Tranco ID might be different if you use a different list than [KJ49W](https://tranco-list.eu/list/KJ49W/full)).

Once completed, the `.pcapng` file can be compressed to safe space.

In [5]:
%%bash
pigz "${DNSCBOR_EVAL_DIR}/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.pcapng"

## Download IoT Datasets

This notebook expects the IoT data sets to be present in the `./04_cbor4dns_eval/input_dataset` directory. The following sub-directories are expected for the datasets of the IoTFinder, MonIoTr, and Yourthings studies, respectively.

- `iotfinder`
- `moniotr`
- `yourthings`

Please read below how to acquire them.

### Download IoTFinder & YourThings Data

You can freely download the datasets IoTFinder and YourThings datasets at https://yourthings.info/data/. The authors ask kindly to cite their work if you use it in your published data.

1. Roberto Perdisci, Thomas Papastergiou, Omar Alrawi, Manos Antonakakis; [IoTFinder: Efficient Large-Scale Identification of IoT Devices via Passive DNS Traffic Analysis](https://doi.org/10.1109/EuroSP48549.2020.00037), in IEEE European Symposium of Security & Privacy, Sept 2020.
2. Omar Alrawi, Chaz Lever, Manos Antonakakis, Fabian Monrose; [SoK: Security Evaluation of Home-Based IoT Deployments](https://doi.org/10.1109/SP.2019.00013), in IEEE Symposium of Security & Privacy, May 2019.

For your convenience, we provided the following script to download the datasets.
Keep in mind that this downloads about 140 Gbytes of compressed data and may thus, depending on your Internet connection, take several hours complete. Over a 50 MBit/s connection it took about 7 hours. We recommend to run it in background, e.g., in a `tmux` session.

In [6]:
list_code(DNSCBOR_EVAL_DIR / "download_iotfinder_and_yourthings.sh")

#!/usr/bin/env bash
#
# collect_dns_hex.sh
# Copyright (C) 2023 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"
INPUT_DATASETS="${SCRIPT_DIR}/input_datasets"

IOTFINDER_URLS=(
    "https://www.dropbox.com/s/3tv44ywzyahy3km/dns_2019_08.tgz?dl=0"
    "https://www.dropbox.com/s/v5pga0qpnywe1et/dns_2019_09.tgz?dl=0"
)
YOURTHINGS_URLS=(
    "https://www.dropbox.com/s/0jltu5o4a9kk7z8/iot_traffic20180320.tgz?dl=0"
    "https://www.dropbox.com/s/xo75oan8juoyb9o/iot_traffic20180321.tgz?dl=0"
    "https://www.dropbox.com/s/2iolbgnaw68cpb0/iot_traffic20180328.tgz?dl=0"
    "https://www.dropbox.com/s/blyem2vfbwbwd4h/iot_traffic20180410.tgz?dl=0"
    "https://www.dropbox.com/s/rcodps77sot88xl/iot_traffic20180411.tgz?dl=0"
    "https://www.dropbox.com/s/ldpkjs799wxf7la/iot_traffic20180412.tgz?dl=0"
    "https://www.dropbox.com/s/z11dgm9u6kuzvml/iot_traffic20180413.tgz?dl=0"
    "https://www.dropbox.com/s/2j62g8qxhby4sqh/iot_traffic20180414.tgz?dl=0"
    "https://www.dropbox.com/s/n6epxutlemcrcrp/iot_traffic20180415.tgz?dl=0"
    "https://www.dropbox.com/s/tfrhq7noobgxpi0/iot_traffic20180416.tgz?dl=0"
    "https://www.dropbox.com/s/gsly960mzi6f0on/iot_traffic20180417.tgz?dl=0"
    "https://www.dropbox.com/s/8klt5f9fr164f5n/iot_traffic20180418.tgz?dl=0"
    "https://www.dropbox.com/s/c0uxli3cirzill1/iot_traffic20180419.tgz?dl=0"
)

download_dataset() {
    name="${1}"
    urls=(${@})
    unset urls[0]

    mkdir -p "${INPUT_DATASETS}/${name}"
    for url in ${urls[@]}; do
        curl -L "$url" | tar -xz -C "${INPUT_DATASETS}/${name}"
    done
}

download_dataset "iotfinder" "${IOTFINDER_URLS[@]}"
download_dataset "yourthings" "${YOURTHINGS_URLS[@]}"

In [7]:
%%bash
tmux new-session -s "download_iot" -d "'${DNSCBOR_EVAL_DIR}'/download_iotfinder_and_yourthings.sh"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "download_iot"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "download_iot" C-c
```

### Download MonIoTr

The authors of the MonIoTr study ask you to agree to their data sharing agreement to download their data set. Please see [their website](https://moniotrlab.khoury.northeastern.edu/publications/imc19/) for more details. Once you received the link to their drive, download the `iot-data.tgz` their and drop the unpacked files in `./04_cbor4dns_eval/input_dataset/moniotr` manually (with `DNSCBOR_EVAL_DIR` set to `./04_cbor4dns_eval`).

```sh
mkdir -p ${DNSCBOR_EVAL_DIR}/input_datasets/moniotr
tar -C ${DNSCBOR_EVAL_DIR}/input_datasets/moniotr -xzf "<path to downloaded iot-data.tgz>"
```

### Checking Downloaded Datsets

Now, all datasets should be in `./04_cbor4dns_eval/input_dataset`. You can check the correctness with the following command.

In [8]:
!ls "{DNSCBOR_EVAL_DIR}/input_datasets/"
!cd "{DNSCBOR_EVAL_DIR}/input_datasets/" && sha256sum --status -c input_datasets.sha256sum && echo "Input datasets OK" || echo "Error checking dataset consistency" >&2

input_datasets.sha256sum  iotfinder  moniotr  yourthings
Input datasets OK


## Extract DNS Data from PCAPs

Next we extract all relavent information from the PCAP files and store them in `dns_data_tranco.csv.gz` and `dns_data_iot.csv.gz`, respectively.
Both contain the following fields, most of them corresponding to the [Wireshark/TShark Display Filter](https://www.wireshark.org/docs/dfref/) of the same name (annotated with "See filter" in list below).
For each DNS legal packet in the PCAP files (`dns.count.queries == 1`, replicated `vlan` packets are ignored), there is a record.
Unless stated otherwise, multiple values in the same field are separated by a pipe (`|`).

- `dataset`: The dataset the PCAP was from.
- `pcap`: The path of the PCAP file within `./04_cbor4dns_eval/output_datasets/`.
- `frame.number`: Frame number within the PCAP file.
- `frame.protocols`: Protocols in frame as colon (`:`)-separated list.
- `_ws.col.Source`: IP source address (either IPv4 or IPv6).
- `_ws.col.Destination`: IP destination address (either IPv4 or IPv6).
- `dns.flags.response`: DNS header Q/R flag.
- `dns.qry.type`: Queried record type(s).
- `dns.resp.type`: Record type(s) in the answer, authority, and additional section.
- `dns.retransmit_response_in`: Retransmitted response. Original response at the frame number provided here.
- `dns.response_to`: Response is at the frame number provided here.
- `payload`: DNS payload, starting at the DNS header.
  This is aggregated from `udp.payload`, `tcp.reassembled.data`, and `tcp.payload` (in that order) depending on which is set first.
  For TCP, the preceeding length field is removed.
- `dns.qry.name`: Queried record name(s).
- `dns.resp.name`: Record name(s) in the answer, authority, and additional section.
- `dns.afsdb.hostname`: See filter.
- `dns.cname`: See filter.
- `dns.dname`: See filter.
- `dns.mr`: See filter.
- `dns.ns`: See filter.
- `dns.nsec.next_domain_name`: See filter.
- `dns.ptr.domain_name`: See filter.
- `dns.rrsig.signers_name`: See filter.
- `dns.rt.intermediate_host`: See filter.
- `dns.soa.mname`: See filter.
- `dns.soa.rname`: See filter.
- `dns.srv.name`: See filter.
- `dns.svcb.targetname`: See filter.
- `dns.winsr.name_result_domain`: See filter.
- `dns.a`: See filter.
- `dns.aaaa`: See filter.
- `dns.apl.afdpart.ipv4`: See filter.
- `dns.apl.afdpart.ipv6`: See filter.
- `dns.ilnp.l32`: See filter.
- `dns.ipseckey.gateway_ipv4`: See filter.
- `dns.ipseckey.gateway_ipv6`: See filter.
- `dns.svcb.svcparam.ipv4hint.ip`: See filter.
- `dns.svcb.svcparam.ipv6hint.ip`: See filter.
- `dns.wins.wins_server`: See filter.
- `dns.wks.address`: See filter.
- `dns.xpf.destination_ipv4`: See filter.
- `dns.xpf.destination_ipv6`: See filter.
- `dns.xpf.source_ipv4`: See filter.
- `dns.xpf.source_ipv6`: See filter.
- `dns.query_payload`: The `payload` of the frame identified by `dns.respnose_to`.
- `qr-diff`: Tuple of the difference between `dns.response_to` and `frame.number` and the difference between `dns.retransmit_response_in` and `frame.number`.
- `stored`: Debug field for the number of responses stored to associate them to later queries.


### Tranco

In [9]:
list_code(DNSCBOR_EVAL_DIR / "collect_tranco_dns.py")

#! /usr/bin/env python3
#
# Copyright (C) 2024 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import collections
import csv
import multiprocessing
import os
import pathlib
import subprocess
import sys
import traceback

import bounded_pool_executor

import cbor4dns_utils


SCRIPT_PATH = pathlib.Path(__file__).resolve().parent
INPUT_DATASETS = SCRIPT_PATH / "input_datasets"
FIELDS = [
    "frame.number",
    "frame.protocols",
    "_ws.col.Source",
    "_ws.col.Destination",
    "dns.flags.response",
    "dns.qry.type",
    "dns.resp.type",
    "dns.retransmit_response_in",
    "dns.response_to",
    "udp.payload",
    "tcp.reassembled.data",
    "tcp.payload",
    "dns.qry.name",
    "dns.resp.name",
    "dns.afsdb.hostname",
    "dns.cname",
    "dns.dname",
    "dns.mr",
    "dns.ns",
    "dns.nsec.next_domain_name",
    "dns.ptr.domain_name",
    "dns.rrsig.signers_name",
    "dns.rt.intermediate_host",
    "dns.soa.mname",
    "dns.soa.rname",
    "dns.srv.name",
    "dns.svcb.targetname",
    "dns.winsr.name_result_domain",
    "dns.a",
    "dns.aaaa",
    "dns.apl.afdpart.ipv4",
    "dns.apl.afdpart.ipv6",
    "dns.ilnp.l32",
    "dns.ipseckey.gateway_ipv4",
    "dns.ipseckey.gateway_ipv6",
    "dns.svcb.svcparam.ipv4hint.ip",
    "dns.svcb.svcparam.ipv6hint.ip",
    "dns.wins.wins_server",
    "dns.wks.address",
    "dns.xpf.destination_ipv4",
    "dns.xpf.destination_ipv6",
    "dns.xpf.source_ipv4",
    "dns.xpf.source_ipv6",
]
QUERIES = None
RESPONSES = None
WRITER = None
DATASET = None


def rewrite_row(row):
    global DATASET, QUERIES, RESPONSES, WRITER
    row.update(DATASET)
    try:
        if not row["udp.payload"] and not row["tcp.reassembled.data"]:
            row["payload"] = row["tcp.payload"][4:]
        elif not row["udp.payload"]:
            row["payload"] = row["tcp.reassembled.data"][4:]
        else:
            row["payload"] = row["udp.payload"]
        row.pop("udp.payload", None)
        row.pop("tcp.reassembled.data", None)
        row.pop("tcp.payload", None)
        frame_number = int(row["frame.number"])
        if row["dns.flags.response"] == "True" or row["dns.flags.response"] == "1":
            if row["dns.retransmit_response_in"]:
                retransmit_response_in = int(row["dns.retransmit_response_in"])
                try:
                    if (
                        retransmit_response_in in RESPONSES
                        and RESPONSES[retransmit_response_in] in QUERIES
                    ):
                        row["dns.query_payload"] = QUERIES[
                            RESPONSES[retransmit_response_in]
                        ]
                except KeyError as e:
                    print(traceback.format_exc(), file=sys.stderr)
                    print("Error:", e, "on", row, file=sys.stderr)
                row["qr-diff"] = (
                    f"{frame_number - retransmit_response_in}|"
                    f"{frame_number - RESPONSES.get(retransmit_response_in, 0)}"
                )
            if row["dns.response_to"]:
                response_to = int(row["dns.response_to"])
                RESPONSES[frame_number] = response_to
                if response_to in QUERIES:
                    try:
                        row["dns.query_payload"] = QUERIES[response_to]
                    except KeyError as e:
                        print(traceback.format_exc(), file=sys.stderr)
                        print("Error:", e, "on", row, file=sys.stderr)
                row["qr-diff"] = f"{frame_number - response_to}"
            row["stored"] = len(RESPONSES)
        else:
            QUERIES[frame_number] = row["payload"]
            row["stored"] = len(QUERIES)
        WRITER.writerow(row)
    except Exception as e:
        print(traceback.format_exc(), file=sys.stderr)
        print("Error:", e, "on", row, file=sys.stderr)
        sys.stderr.flush()


def main():
    global DATASET, QUERIES, RESPONSES, WRITER
    parser = argpa

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command (with the UV setup you might need to step into the virtualenv first, adding, e.g., `. '${DNSCBOR_EVAL_DIR}'/../.env/bin/activate;` before the `${DNSCBOR_EVAL_DIR}/collect_tranco_dns.py`):

In [10]:
%%bash
tmux new-session -s "collect_tranco" -d \
    "'${DNSCBOR_EVAL_DIR}/collect_tranco_dns.py' \
        '${DNSCBOR_EVAL_DIR}/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.pcapng.gz' | \
            pigz > '${DNSCBOR_EVAL_DIR}/output_datasets/dns_data_tranco.csv.gz'"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "collect_tranco"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "collect_tranco" C-c
```

### IoT

In [11]:
list_code(DNSCBOR_EVAL_DIR / "collect_iot_dns.sh")

#!/usr/bin/env bash
#
# collect_dns_hex.sh
# Copyright (C) 2023 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

PROCS=$(grep -c '^processor' /proc/cpuinfo)
FIELDS='-e frame.number -e frame.protocols -e _ws.col.Source -e _ws.col.Destination
    -e dns.flags.response -e dns.qry.type -e dns.resp.type -e dns.retransmit_response_in -e dns.response_to
    -e udp.payload -e tcp.reassembled.data -e tcp.payload
    -e dns.qry.name -e dns.resp.name
    -e dns.afsdb.hostname -e dns.cname -e dns.dname -e dns.mr -e dns.ns
    -e dns.nsec.next_domain_name -e dns.ptr.domain_name -e dns.rrsig.signers_name
    -e dns.rt.intermediate_host -e dns.soa.mname -e dns.soa.rname -e dns.srv.name
    -e dns.svcb.targetname -e dns.winsr.name_result_domain
    -e dns.a -e dns.aaaa -e dns.apl.afdpart.ipv4 -e dns.apl.afdpart.ipv6 -e dns.ilnp.l32
    -e dns.ipseckey.gateway_ipv4 -e dns.ipseckey.gateway_ipv6 -e dns.svcb.svcparam.ipv4hint.ip
    -e dns.svcb.svcparam.ipv6hint.ip -e dns.wins.wins_server -e dns.wks.address
    -e dns.xpf.destination_ipv4 -e dns.xpf.destination_ipv6 -e dns.xpf.source_ipv4
    -e dns.xpf.source_ipv6'
INPUT_DATASETS="${SCRIPT_DIR}/input_datasets"
OUTPUT_DATASETS="${SCRIPT_DIR}/output_datasets"

mkdir -p "${OUTPUT_DATASETS}"

parse_pcaps() {
    pcap="${1}"
    dataset_pcap="$(echo "${pcap}" | sed "s#${INPUT_DATASETS}/##")"
    dataset=$(echo "${dataset_pcap}" | cut -d'/' -f1)
    tshark -Y "dns.count.queries == 1 && !vlan" -r "${pcap}" -Tfields ${FIELDS} \
        -E aggregator='|' -E separator=';' | \
        gawk -f ${SCRIPT_DIR}/collect_dns.awk | sed "s#^#${dataset};${dataset_pcap};#"
}

export -f parse_pcaps
export FIELDS
export INPUT_DATASETS
export SCRIPT_DIR

HEADER=$(
    echo "${FIELDS}" | tr -d '\n' |
        sed -E -e 's/-e\s+//' -e 's/\s+-e\s+/;/g' -e 's/;tcp[^;]+//g' -e 's/udp.payload/payload/g' | \
        xargs printf "dataset;pcap;%s;dns.query_payload;qr-diff;stored\n"
)
find ${INPUT_DATASETS}/{yourthings,iotfinder,moniotr}/ \
    -name "*.pcap" -o -name "eth1-*" -o -name "*.pcapng" -o -name "*.pcapng.gz" | \
    parallel --line-buffer -j"${PROCS}" --progress --eta parse_pcaps > "${OUTPUT_DATASETS}/dns_data_iot.csv"
sed -i "1i\
${HEADER}
" "${OUTPUT_DATASETS}/dns_data_iot.csv"

In [12]:
list_code(DNSCBOR_EVAL_DIR / "collect_dns.awk")

BEGIN {
    FS=OFS=";"
    queries[0] = ""
    responses[0] = ""
}
{
    if (!$10 && !$11) {
        # use tcp.payload as payload (without leading length)
        $10 = substr($12,5)
    }
    else if (!$10) {
        # use tcp.reassembled.data as payload (without leading length)
        $10 = substr($11, 5)
    }
    # else use udp.payload as length
    # move the rest two columns to the left
    for (i = 11; i < (NF - 2); i++) {
        $i = $(i + 2)
    }
    # and remove the two tcp.payload/tcp.reassembled.data column
    NF -= 2
    # find dns.query_payloads:
    $(NF + 1) = ""
    $(NF + 1) = ""
	if ($5 == "True" || $5 == 1) { # is response
        if ($8) {  # dns.retransmit_response_in has a value
            if (responses[$8] && queries[responses[$8]]) {
                $(NF - 1) = queries[responses[$8]]
            }
            $NF = ($1 - $8) "|" ($1 - responses[$8])
        }
        if ($9) { # dns.response_to has a value
            # store dns.response_to under frame.number
            responses[$1] = $9
            if (queries[$9]) {
                $(NF - 1) = queries[$9]
            }
            $NF = $1 - $9
        }
        # Remove older responses to safe memory
        while (length(responses) > 500000) {
            min = 0;
            for (key in responses) {  # could use length here, but we need to search a minimum
                if (!min || (key < min)) {
                    min = key;
                }
            }
            delete responses[min];
        }
        $(NF + 1) = length(responses)
    }
    else { # is query
        # store query payload under frame.number
        queries[$1] = $10

        # Remove older queries to safe memory
        while (length(queries) > 500000) {
            min = 0;
            for (key in queries) {  # could use length here, but we need to search a minimum
                if (!min || (key < min)) {
                    min = key;
                }
            }
            delete queries[min];
        }
        $(NF + 1) = length(queries)
    }
    # add column for dns.query_payload
    print $0
}

To start a detached TMUX session running this script **in background of the Docker setup**, run the following command. It assumes that GNU awk, `gawk`, is installed (as pre-installed in the Docker setup).

In [13]:
%%bash
tmux new-session -s "collect_iot" -d \
    "'${DNSCBOR_EVAL_DIR}/collect_iot_dns.sh' && \
        pigz '${DNSCBOR_EVAL_DIR}'/output_datasets/dns_data_iot.csv"

To attach that TMUX session, run the following commant in a Terminal in your Jupyter Lab.

```sh
tmux attach -t "collect_iot"
```

To kill the TMUX session you can use the following command into a Terminal in your Jupyter Lab.

```sh
tmux send-keys -t "collect_iot" C-c
```